# 👥 Enterprise Employee Attrition & Risk Scoring Analysis Notebook
This notebook performs end-to-end data cleaning, XGBoost modeling, 5-Fold Stratified Cross-Validation, PR-AUC evaluation, hyperparameter tuning, threshold optimization, error analysis, SHAP explainability, and employee risk score calculation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import joblib
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    roc_curve, confusion_matrix, classification_report, f1_score,
    precision_score, recall_score
)
from xgboost import XGBClassifier
import shap

print("Libraries successfully imported!")

## Clean & Fix Dataset

In [ ]:
df = pd.read_csv("HR_Attrition.csv")
cols_to_drop = [c for c in ["EmployeeCount", "Over18", "StandardHours", "EmployeeNumber"] if c in df.columns]
df_clean = df.drop(columns=cols_to_drop)
df_clean["Attrition_Target"] = df_clean["Attrition"].map({"Yes": 1, "No": 0})
X = df_clean.drop(columns=["Attrition", "Attrition_Target"])
y = df_clean["Attrition_Target"]
cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
print(f"Shape: {X.shape}, Target ratio: {y.mean():.4f}")

##5-Fold Stratified Cross Validation with XGBoost

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros(len(X))
preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"), cat_cols),
    ("num", "passthrough", num_cols)
])
ratio = (len(y) - sum(y)) / sum(y)
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
    X_train_trans = preprocessor.fit_transform(X_train)
    X_val_trans = preprocessor.transform(X_val)
    model = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=4, scale_pos_weight=ratio, random_state=42, eval_metric="logloss")
    model.fit(X_train_trans, y_train)
    oof_probs[val_idx] = model.predict_proba(X_val_trans)[:, 1]
overall_pr_auc = average_precision_score(y, oof_probs)
overall_roc_auc = roc_auc_score(y, oof_probs)
print(f"OOF PR-AUC: {overall_pr_auc:.4f}, OOF ROC-AUC: {overall_roc_auc:.4f}")

##Threshold Optimization

In [ ]:
thresholds = np.linspace(0.05, 0.95, 91)
best_f1, best_thresh = 0.0, 0.5
for t in thresholds:
    preds = (oof_probs >= t).astype(int)
    score = f1_score(y, preds, zero_division=0)
    if score > best_f1:
        best_f1, best_thresh = score, t
print(f"Optimal Decision Threshold: {best_thresh:.2f} -> Max F1 Score: {best_f1:.4f}")

## SHAP Explainability & Risk Scoring

In [ ]:
X_trans = preprocessor.fit_transform(X)
model.fit(X_trans, y)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_trans)
print("SHAP analysis complete!")
risk_scores = (oof_probs * 100).round(2)
risk_tiers = pd.cut(risk_scores, bins=[-1, 35, 70, 100], labels=["Low Risk", "Medium Risk", "High Risk"])
print(risk_tiers.value_counts())